In [1]:
!pip install transformers
!pip install spacy
!pip install scispacy
!pip install langchain
!pip install torch

     ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
     -------- ------------------------------- 0.3/1.3 MB ? eta -:--:--
     ---------------- ----------------------- 0.5/1.3 MB 1.1 MB/s eta 0:00:01
     ------------------------ --------------- 0.8/1.3 MB 1.3 MB/s eta 0:00:01
     -------------------------------- ------- 1.0/1.3 MB 1.3 MB/s eta 0:00:01
     ---------------------------------------- 1.3/1.3 MB 1.4 MB/s  0:00:01
  Installing build dependencies: started
  Installing build dependencies: still running...
  Installing build dependencies: finished with status 'error'


  error: subprocess-exited-with-error
  
  installing build dependencies for spacy did not run successfully.
  exit code: 1
  
  [147 lines of output]
  Ignoring numpy: markers 'python_version < "3.9"' don't match your environment
    Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
    Using cached Cython-0.29.37-py2.py3-none-any.whl.metadata (3.1 kB)
    Using cached cymem-2.0.13-cp313-cp313-win_amd64.whl.metadata (9.9 kB)
    Using cached preshed-3.0.12-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
    Using cached murmurhash-1.0.15-cp313-cp313-win_amd64.whl.metadata (2.3 kB)
    Installing build dependencies: started
    Installing build dependencies: still running...
    Installing build dependencies: finished with status 'error'
    error: subprocess-exited-with-error
  
    installing build dependencies for thinc did not run successfully.
    exit code: 1
  
    [121 lines of output]
    Ignoring numpy: markers 'python_version < "3.9"' don't match your environment

  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
Using cached torch-2.10.0-cp313-cp313-win_amd64.whl (113.8 MB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.22 requires torchvision, which is not installed.
nnunetv2 2.6.3 requires torch<2.9.0,>=2.1.2, but you have torch 2.10.0 which is incompatible.


In [1]:
import spacy
import torch
from transformers import pipeline

C:\Users\sinan\anaconda3\envs\medical_ai\lib\site-packages\spacy\cli\info.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
C:\Users\sinan\anaconda3\envs\medical_ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nlp = spacy.load("en_core_sci_sm")

In [3]:
def extract_medical_entities(text):

    doc = nlp(text)

    entities = []

    for ent in doc.ents:
        entities.append((ent.text, ent.label_))

    return entities

In [4]:
medical_dictionary = {

    "myocardial infarction": "heart attack",

    "hypertension": "high blood pressure",

    "dyspnea": "difficulty breathing",

    "tachycardia": "fast heart rate",

    "troponin": "protein released when heart muscle is damaged"

}

In [7]:
!pip install --upgrade transformers

In [9]:
from transformers import pipeline

simplifier = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 282/282 [00:00<00:00, 2346.47it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', '

In [10]:
def simplify_report(text):

    prompt = f"""
    Explain the following medical report in simple language for a patient.

    Medical Report:
    {text}
    """

    result = simplifier(prompt, max_length=200)

    return result[0]['generated_text']

In [11]:
critical_conditions = [
    "stroke",
    "heart attack",
    "myocardial infarction",
    "cardiac arrest"
]

def detect_risk(text):

    for condition in critical_conditions:

        if condition in text.lower():

            return "⚠ Serious condition detected. Immediate medical attention recommended."

    return "No critical condition detected."

In [12]:
def medical_report_simplifier(report):

    entities = extract_medical_entities(report)

    simplified = simplify_report(report)

    risk = detect_risk(report)

    return {
        "entities": entities,
        "simplified_explanation": simplified,
        "risk_warning": risk
    }

In [13]:
report = """
Patient presents with dyspnea and tachycardia.
ECG shows abnormal ST elevation and elevated troponin levels.
"""

result = medical_report_simplifier(report)

result

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'entities': [('Patient', 'ENTITY'),
  ('dyspnea', 'ENTITY'),
  ('tachycardia', 'ENTITY'),
  ('ECG', 'ENTITY'),
  ('abnormal', 'ENTITY'),
  ('ST', 'ENTITY'),
  ('elevation', 'ENTITY'),
  ('elevated', 'ENTITY'),
  ('troponin levels', 'ENTITY')],
 'simplified_explanation': '\n    Explain the following medical report in simple language for a patient.\n\n    Medical Report:\n    \nPatient presents with dyspnea and tachycardia.\nECG shows abnormal ST elevation and elevated troponin levels.\n\n    ',
 'risk_warning': 'No critical condition detected.'}

In [14]:
print("Medical Entities Detected:")
print(result["entities"])

print("\nSimplified Explanation:")
print(result["simplified_explanation"])

print("\nRisk Warning:")
print(result["risk_warning"])

Medical Entities Detected:
[('Patient', 'ENTITY'), ('dyspnea', 'ENTITY'), ('tachycardia', 'ENTITY'), ('ECG', 'ENTITY'), ('abnormal', 'ENTITY'), ('ST', 'ENTITY'), ('elevation', 'ENTITY'), ('elevated', 'ENTITY'), ('troponin levels', 'ENTITY')]

Simplified Explanation:

    Explain the following medical report in simple language for a patient.

    Medical Report:
    
Patient presents with dyspnea and tachycardia.
ECG shows abnormal ST elevation and elevated troponin levels.

    

Risk Warning:
No critical condition detected.


In [15]:
!pip install streamlit

  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl.metadata (19 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached protobuf-6.33.5-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached pyarrow-23.0.1-cp310-cp310-win_amd64.whl.metadata (3.1 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jsonschema_specifications-2025.9.1-p